In [2]:
# -------------------------------------------------------------
#  Code for: "Introduction to Integer Programming and Applications with Julia"
#  Chapter: 8 – Branch and Bound
#  Section: Exercise 2
#  Author(s): Luiz Henrique Nogueira Lorena
# -------------------------------------------------------------

using CSV, DataFrames 
using LinearAlgebra, Geodesy, DataStructures # For PriorityQueue
using Graphs, MetaGraphs
using Makie, CairoMakie, GraphMakie, NetworkLayout
using CairoMakie: Legend, PolyElement
using PyCall

# --- Core Data Structures ---

struct PMedianProblem
    distances::Matrix{Float64}
    p::Int
    df_clients::DataFrame
    df_facilities::DataFrame
end

# --- Helper function to plot the solution ---

function plot_bnb_pmedian_solution(problem::PMedianProblem, graph::MetaDiGraph)
    
    # Import folium library and create map
    folium = pyimport("folium")
    plugins = pyimport("folium.plugins")
    m = folium.Map(tiles="Cartodb Positron")

    bounds = [(minimum(problem.df_clients[:,1]), 
               minimum(problem.df_clients[:,2])), 
              (maximum(problem.df_clients[:,1]), 
               maximum(problem.df_clients[:,2]))]
    m.fit_bounds(bounds)

    # For three colors
    colors = ["green", 
              "orange", 
              "purple"]

    # Add facility points
    color_id = 1
    for (id,row) in enumerate(eachrow(problem.df_facilities))        
        lat = round(row.latitude, digits=4)
        lon = round(row.longitude, digits=4)
        color = "black"
        tooltip = "Facility $id (closed)"
        if id in get_prop(graph, :best_ub_sol)
            tooltip = "Facility $id (open)"
            color = colors[color_id]
            color_id += 1
        end
        folium.Marker([lat, lon],
                      tooltip=tooltip,
                      icon=folium.Icon(color=color, icon="fa-building", prefix="fa")).add_to(m)
    end

    # Extract the best solution assignment
    facilities_labels = sort(collect(get_prop(graph, :best_ub_sol)))
    ids = argmin(problem.distances[:, facilities_labels],dims=2)
    client_colors = [colors[id[2]] for id in ids]

    for (id,row) in enumerate(eachrow(problem.df_clients))
        lat = round(row.latitude, digits=4)
        lon = round(row.longitude, digits=4)
        color = client_colors[id]
        folium.CircleMarker([lat, lon],
                            radius=2,
                            color=color,
                            fill_color=color,
                            fill_opacity=0.7,
                            tooltip="Client $id").add_to(m)
    end
    return m
end

# --- Helper function to load problem data ---

function load_problem_data(file_clients::String, file_facilities::String; p=3)
    # Load dataset
    df_clients = CSV.read("data/pmp_clients.csv", DataFrame)
    df_facilities = CSV.read("data/pmp_facilities.csv", DataFrame)
    # Create distance matrix
    d = zeros(nrow(df_clients), nrow(df_facilities))
    for i in 1:nrow(df_clients)
        for j in 1:nrow(df_facilities)
            d[i, j] = Geodesy.euclidean_distance(
                Geodesy.LatLon(df_clients.latitude[i], df_clients.longitude[i]),
                Geodesy.LatLon(df_facilities.latitude[j], df_facilities.longitude[j])
            )
        end
    end
    num_clients, num_facilities = size(d)
    return PMedianProblem(d, p, df_clients, df_facilities)
end

# --- Helper, Initialization, and Lagrangian Functions ---

function calculate_total_cost(problem::PMedianProblem, open_facilities::Set)
    isempty(open_facilities) && return Inf
    # For each client, find the minimum distance to an open facility and sum them up.
    return sum(minimum(problem.distances[c, f] for f in open_facilities) for c in 1:size(problem.distances, 1))
end

# Optimized greedy heuristic.
function greedy_upper_bound(problem::PMedianProblem)
    
    num_clients, num_facilities = size(problem.distances)
    
    # Start with no facilities open. The cost for each client is effectively infinite.
    min_dists = fill(Inf, num_clients)
    open_facilities = Set{Int}()
    
    # Iteratively add the facility that provides the greatest cost reduction.
    for _ in 1:problem.p
        best_f, max_reduction = -1, -Inf
        
        # Find the best facility to add from the remaining candidates.
        for f in setdiff(Set(1:num_facilities), open_facilities)
            # Calculate the cost reduction this facility would provide.
            reduction = sum(max(0.0, min_dists[c] - problem.distances[c, f]) for c in 1:num_clients)
            if reduction > max_reduction
                max_reduction = reduction
                best_f = f
            end
        end
        
        # Add the best facility and update the minimum distances for each client.
        if best_f != -1
            push!(open_facilities, best_f)
            for c in 1:num_clients
                min_dists[c] = min(min_dists[c], problem.distances[c, best_f])
            end
        else
            break # No more facilities to add or no improvement possible.
        end
    end
    
    final_cost = sum(min_dists)
    return final_cost, open_facilities
end

# Optimized Lagrangian function using vectors instead of dicts.
function lagrangian_lower_bound(problem::PMedianProblem, fixed_open::Set, fixed_closed::Set, global_ub::Float64)
    num_clients, num_facilities = size(problem.distances)
    max_iterations = 30; alpha = 1.0
    lambdas = zeros(num_clients)
    best_lb = -Inf; best_y = Set{Int}()
    
    available = setdiff(Set(1:num_facilities), fixed_closed)
    
    for _ in 1:max_iterations
        # Calculate profit for each available facility.
        profits = Dict(j => sum(min(0.0, problem.distances[i, j] - lambdas[i]) for i in 1:num_clients) for j in available)
        
        p_select = problem.p - length(fixed_open)
        
        # Sort available, non-fixed-open facilities by profit.
        candidates = setdiff(available, fixed_open)
        sorted_f = sort(collect(candidates), by=f->get(profits, f, 0.0))
        
        y_star = copy(fixed_open)
        if p_select > 0 && !isempty(sorted_f)
            union!(y_star, sorted_f[1:min(p_select, length(sorted_f))])
        end
        
        # Calculate the lower bound.
        lb = sum(lambdas) + sum(get(profits, j, 0.0) for j in y_star)
        if lb > best_lb 
            best_lb = lb 
            best_y = y_star 
        end
        
        # Update subgradient and step size.
        subgrad = [1.0 - (any(j -> problem.distances[i, j] < lambdas[i], y_star) ? 1 : 0) for i in 1:num_clients]
        norm_sq = sum(abs2, subgrad)
        
        if norm_sq < 1e-12 || abs(global_ub - lb) < 1e-6 
            break 
        end
        step = alpha * (global_ub - lb) / norm_sq
        
        # Update lambdas.
        lambdas .= max.(0.0, lambdas .+ step .* subgrad)
    end
    return best_lb, best_y
end

# --- SOLVER: solve the BnB ---

function solve_bnb_pmedian(problem::PMedianProblem)
    num_facilities = size(problem.distances, 2)
    
    # Find initial upper and lower bounds (root node).
    ub, ub_solution = greedy_upper_bound(problem)
    lb, lb_solution = lagrangian_lower_bound(problem, Set(), Set(), ub)
    graph = MetaDiGraph()
    add_vertex!(graph)
    set_props!(graph, 1, Dict(
        :ub => ub,
        :lb => lb,
        :fixed_open => Set(),
        :fixed_close => Set(),
        :state => :explored
    ))
    set_prop!(graph, :best_id, 1)
    set_prop!(graph, :best_ub, ub)
    set_prop!(graph, :best_ub_sol, ub_solution)

    # Use a PriorityQueue for best-first search (lower LB is higher priority).
    active_nodes = PriorityQueue{Int, Float64}()
    enqueue!(active_nodes, 1 => lb)

    while !isempty(active_nodes)
        current_node, current_lb = dequeue_pair!(active_nodes)
        current_props = props(graph, current_node)

        # Pruning by bounding
        if current_lb >= get_prop(graph, :best_ub)
            current_props[:state] = :pruned_by_bounding
            set_props!(graph, current_node, current_props)
            continue
        end

        # Pruning by integrality
        current_fixed_open = get_prop(graph, current_node, :fixed_open)
        current_fixed_close = get_prop(graph, current_node, :fixed_close)
        unfixed = setdiff(Set(1:num_facilities), current_fixed_open, current_fixed_close)
        if isempty(unfixed)
            current_props[:state] = :pruned_by_integrality
            set_props!(graph, current_node, current_props) 
            continue
        end

        # Branching on the first unfixed facility
        branching_facility = first(unfixed)

        # Branch on closing or opening the selected facility.
        for decision in (:close, :open)
            # Create child node based on decision.
            new_fixed_open = decision == :open ? union(current_fixed_open, [branching_facility]) : current_fixed_open
            new_fixed_close = decision == :close ? union(current_fixed_close, [branching_facility]) : current_fixed_close

            # Pre-pruning based on feasibility.
            if length(new_fixed_open) > problem.p || num_facilities - length(new_fixed_close) < problem.p
                continue
            end
            
            # Compute the lower bound for the child node.
            child_lb, y_star = lagrangian_lower_bound(problem, new_fixed_open, new_fixed_close, get_prop(graph, :best_ub))
            
            # Add node and edge to the history graph.
            add_vertex!(graph)
            child_id = nv(graph)
            node_props = Dict{Symbol, Any}(
                :ub => get_prop(graph, :best_ub),
                :lb => child_lb,
                :fixed_open => new_fixed_open, 
                :fixed_close => new_fixed_close,
                :state => :explored 
            )
            # Check if this node provides a new, better feasible solution.
            if length(y_star) == problem.p
                cost = calculate_total_cost(problem, y_star)
                if cost < get_prop(graph, :best_ub)
                    set_prop!(graph, :best_id, child_id)
                    set_prop!(graph, :best_ub, cost)
                    set_prop!(graph, :best_ub_sol, y_star)
                    node_props[:state] = :pruned_by_integrality
                    node_props[:ub] = cost
                end
            end            
            set_props!(graph, child_id, node_props)
            add_edge!(graph, current_node, child_id)
            set_prop!(graph, current_node, child_id, :label, "$(decision == :open ? "Open" : "Close") F$(branching_facility)")
            enqueue!(active_nodes, child_id => child_lb)
        end
    end

    # After the search, update the :state of best node to :optimal
    best_id = get_prop(graph, :best_id)
    best_props = props(graph, best_id)
    best_props[:state] = :optimal
    set_props!(graph, best_id, best_props)

    return graph
end

# --- Main Execution Block ---
problem = load_problem_data("data/pmp_clients.csv", "data/pmp_facilities.csv", p = 3)

# Solve the problem using branch-and-bound
graph = solve_bnb_pmedian(problem)

println("Total nodes created in the BnB graph: ", nv(graph))
println("Best node id: ", get_prop(graph, :best_id))
println("Best objective found: ", get_prop(graph, :best_ub), " meters")
println("Best solution: ", get_prop(graph, :best_ub_sol))

# Plot the branch-and-bound solution on map
plot_bnb_pmedian_solution(problem, graph)

Total nodes created in the BnB graph: 252
Best node id: 15
Best objective found: 640019.8421838199 meters
Best solution: Set(Any[4, 13, 8])


PyObject <folium.folium.Map object at 0x0000026D62E58DD0>